## Setup
Ejecuta primero la celda de instalacion para tener todas las dependencias listas.

In [1]:
# Instalacion de dependencias necesarias para la practica
%pip install -q openai azure-identity python-dotenv

Note: you may need to restart the kernel to use updated packages.



## 2) Experimentación con Parámetros del Modelo

**Objetivo:** Entender cómo los parámetros del modelo (temperature, top_p, penalties) modifican el comportamiento de las respuestas generadas.

### 2.1 - Experimentación con temperature
Prueba el **mismo prompt** con diferentes valores de `temperature`:
- `temperature = 0.0`
- `temperature = 0.5`
- `temperature = 1.0`
- `temperature = 1.5` (si la API lo permite)

Analiza cómo cambian las respuestas. Usa un caso práctico (ejemplo: generar un eslogan, escribir código, responder una pregunta técnica).

In [3]:
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

project_endpoint = os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT")
if not project_endpoint:
    raise ValueError("Falta AZURE_FOUNDRY_PROJECT_ENDPOINT en el entorno (.env).")

base_url = project_endpoint.rstrip("/") + "/openai/v1"

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=base_url,
    api_key=token_provider(),
)

# Caso práctico: Generación de eslóganes creativos
prompt = '''
Escribe 3 eslóganes cortos y llamativos para una nueva marca de tazas de café inteligentes llamadas "ThermoMug", que mantienen la bebida a la temperatura perfecta durante todo el día.
'''

chat_model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")

# Valores de temperatura para experimentar
temperatures = [0.0, 0.5, 1.0, 1.5]

for temp in temperatures:
    print(f"\n{'='*50}")
    print(f"PROBANDO CON TEMPERATURE: {temp}")
    print(f"{'='*50}")
    
    
    response = client.responses.create(
        model=chat_model,
        input=prompt,
        temperature=temp # <-- Aquí inyectamos la temperatura
    )

    print(f"Response:\n{response.output_text}\n")
    print(f"Status:   {response.status}")
    print(f"Output tokens: {response.usage.output_tokens}")
        
  


PROBANDO CON TEMPERATURE: 0.0
Response:
1. **"ThermoMug: Sabor perfecto, temperatura ideal."**

2. **"Tu café, siempre a la temperatura de tu amor."**

3. **"ThermoMug: Disfruta cada sorbo, sin prisa."**

Status:   completed
Output tokens: 55

PROBANDO CON TEMPERATURE: 0.5
Response:
¡Claro! Aquí tienes tres eslóganes para "ThermoMug":

1. **"Sabor perfecto, temperatura ideal: ¡ThermoMug siempre a tu lado!"**

2. **"Café caliente, sonrisas constantes: ¡Descubre ThermoMug!"**

3. **"Tu café, tu temperatura: ¡Todo el día con ThermoMug!"**

Status:   completed
Output tokens: 83

PROBANDO CON TEMPERATURE: 1.0
Response:
Claro, aquí tienes tres eslóganes para "ThermoMug":

1. **"Sabor constante, sorbo perfecto."**
2. **"Tu café, a la temperatura ideal, todo el día."**
3. **"Con ThermoMug, cada taza es una experiencia."**

Status:   completed
Output tokens: 64

PROBANDO CON TEMPERATURE: 1.5
Response:
¡Claro! Aquí tienes tres eslóganes para "ThermoMug":

1. **“Tu café, tu temperatura: siempre 

### 2.2 - Experimentación con top_p
Prueba el **mismo prompt** con diferentes valores de `top_p`:
- `top_p = 0.1`
- `top_p = 0.5`
- `top_p = 0.9`
- `top_p = 1.0`

Mantén `temperature = 1.0` para ver el efecto puro de top_p. Compara con los resultados de temperature.


In [4]:
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

project_endpoint = os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT")
if not project_endpoint:
    raise ValueError("Falta AZURE_FOUNDRY_PROJECT_ENDPOINT en el entorno (.env).")

base_url = project_endpoint.rstrip("/") + "/openai/v1"

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=base_url,
    api_key=token_provider(),
)

# Mantenemos el mismo caso práctico
prompt = '''
Escribe 3 eslóganes cortos y llamativos para una nueva marca de tazas de café inteligentes llamadas "ThermoMug", que mantienen la bebida a la temperatura perfecta durante todo el día.
'''

chat_model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")

# Valores de top_p para experimentar
top_p_values = [0.1, 0.5, 0.9, 1.0]

for p in top_p_values:
    print(f"\n{'='*50}")
    print(f"PROBANDO CON TOP_P: {p}")
    print(f"{'='*50}")
    
   
    response = client.responses.create(
        model=chat_model,
        input=prompt,
        temperature=1.0,
        top_p=p          
    )

    print(f"Response:\n{response.output_text}\n")
    print(f"Status:   {response.status}")
    print(f"Output tokens: {response.usage.output_tokens}")
        
    


PROBANDO CON TOP_P: 0.1
Response:
1. **"ThermoMug: Sabor perfecto, temperatura ideal."**

2. **"Tu café, siempre a la temperatura de tu amor."**

3. **"ThermoMug: Disfruta cada sorbo, sin prisa."**

Status:   completed
Output tokens: 55

PROBANDO CON TOP_P: 0.5
Response:
1. **"ThermoMug: Sabor a la temperatura perfecta, todo el día."**

2. **"Tu café, siempre caliente. ThermoMug lo hace posible."**

3. **"Disfruta cada sorbo, ThermoMug lo mantiene ideal."**

Status:   completed
Output tokens: 61

PROBANDO CON TOP_P: 0.9
Response:
1. **"ThermoMug: Sabor perfecto, temperatura constante."**

2. **"Disfruta cada sorbo, siempre caliente con ThermoMug."**

3. **"Tu café, a la temperatura ideal, ¡todo el día!"**

Status:   completed
Output tokens: 56

PROBANDO CON TOP_P: 1.0
Response:
¡Claro! Aquí tienes tres eslóganes atractivos para "ThermoMug":

1. **"Sabor siempre a la temperatura ideal."**

2. **"Café caliente, felicidad constante."**

3. **"Termos inteligentes, sorbos perfectos."**

St

### 2.3 - Experimentación con Penalties
Prueba prompts que **tiendan a repetir contenido** (por ejemplo: describir un producto, generar múltiples ideas similares) con:
- `presence_penalty = 0.0` vs `presence_penalty = 0.6`
- `frequency_penalty = 0.0` vs `frequency_penalty = 0.8`
- Combinación de ambos penalties

Analiza qué tipo de repeticiones evita cada uno.

In [ ]:
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

project_endpoint = os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT")
if not project_endpoint:
    raise ValueError("Falta AZURE_FOUNDRY_PROJECT_ENDPOINT en el entorno (.env).")

base_url = project_endpoint.rstrip("/") + "/openai/v1"

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=base_url,
    api_key=token_provider(),
)

# Prompt diseñado para provocar repeticiones y comparar el efecto de penalties
prompt = '''
Describe un nuevo bolígrafo inteligente para estudiantes y da 6 ideas de uso. Mantén un tono publicitario.
'''

chat_model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")

# Configuraciones de penalties para experimentar
penalty_configs = [
    {"presence_penalty": 0.0, "frequency_penalty": 0.0, "label": "Sin penalties"},
    {"presence_penalty": 0.6, "frequency_penalty": 0.0, "label": "Solo presence_penalty"},
    {"presence_penalty": 0.0, "frequency_penalty": 0.8, "label": "Solo frequency_penalty"},
    {"presence_penalty": 0.6, "frequency_penalty": 0.8, "label": "Combinación de ambos"},
]

for cfg in penalty_configs:
    print(f"\n{'='*50}")
    print(f"PROBANDO: {cfg['label']}")
    print(
        f"presence_penalty={cfg['presence_penalty']} | frequency_penalty={cfg['frequency_penalty']}"
    )
    print(f"{'='*50}")

    response = client.chat.completions.create(
        model=chat_model,
        messages=[{"role": "user", "content": prompt}], 
        temperature=1.0,
        presence_penalty=cfg["presence_penalty"],
        frequency_penalty=cfg["frequency_penalty"],
    )

    print(f"Response:\n{response.choices[0].message.content}\n")
    print(f"Output tokens: {response.usage.completion_tokens}")


PROBANDO: Sin penalties
presence_penalty=0.0 | frequency_penalty=0.0
Response:
¡Presentamos el bolígrafo inteligente "SmartPen Edu"! Este innovador bolígrafo está diseñado especialmente para estudiantes que buscan potenciar su experiencia de aprendizaje. Con un estilo moderno y funcional, el SmartPen Edu no solo escribe con precisión, sino que también está equipado con tecnología de punta que transforma la manera en que capturas y organizas tus ideas.

**Características destacadas:**
- **Captura digital:** Registra tus notas a mano y las convierte en formato digital en tiempo real. ¡Adiós a las preocupaciones de perder tus apuntes!
- **Conexión Bluetooth:** Sincroniza automáticamente tus notas con tu tableta o computadora, haciendo que la organización sea pan comido.
- **Audio integrado:** Graba conferencias mientras escribes, asegurando que nunca te pierdas un detalle importante.
- **Modo de traducción:** Con un solo clic, traduce palabras o frases mientras tomas apuntes en varios id

Sin penalties (0.0 / 0.0):
El modelo responde bien, pero se raya un poco repitiendo ideas o palabras.

Solo presence_penalty (0.6 / 0.0):
Mete ideas más nuevas, se nota más variedad y menos “decir lo mismo con otras palabras”.

Solo frequency_penalty (0.0 / 0.8):
Corta bastante la repetición literal de palabras/frases. Queda menos pesado de leer.

Los dos juntos (0.6 / 0.8):
Es la opción más anti-repetición, pero a veces suena un poco menos natural o menos “vendedor”.

### 2.4 - Preguntas Teóricas (Responder con tus propias palabras)
Incluye una sección en el notebook respondiendo estas preguntas basándote en tu experiencia práctica y lo aprendido:

1. **¿Cuál es la diferencia entre top_p y temperature?**
2. **¿Por qué no se recomienda ajustar top_p y temperature al mismo tiempo?**
3. **¿Cuál es la diferencia entre presence_penalty y frequency_penalty?**

1. La temperature ajusta qué tan aleatoria es la elección de palabras, top_p limita la elección al conjunto más probable.
2. No se recomienda ajustar top_p y temperature a la vez porque mezclas dos controles de aleatoriedad y luego es difícil saber cuál cambió realmente la respuesta.
3. Presence penalty penaliza tokens que ya aparecieron y frequency penalty penaliza más cuanto más se repite un token, para reducir repeticiones.

### Entregable (Parte 2):
Notebook `.ipynb` que incluya:
- **Sección de configuración** (imports, credenciales, modelo)
- **Una sección por parámetro experimentado** con:
  - Código mostrando diferentes configuraciones
  - Salidas del modelo para cada configuración
  - Análisis comparativo: ¿cómo afecta cada valor?
- **Sección de preguntas teóricas** con tus respuestas reflexivas (no copiadas)
- **Conclusiones**: ¿Qué configuración usarías para cada tipo de tarea? (código, creatividad, extracción de datos, etc.)

---

## ⭐ Extras

Funcionalidades adicionales que suman puntos:

- **Técnicas adicionales no documentadas:** Investiga y prueba técnicas de prompt engineering que NO aparezcan en `TEORIA.md` pero que uses habitualmente o descubras en tu investigación (cita las fuentes consultadas).

- **Documentación tipo tutorial:** Estructura el notebook como un mini-tutorial que otro estudiante podría seguir para aprender sobre el tema.

---

## Formato y criterios de entrega

- **Formato:** Dos notebooks `.ipynb` separados (uno por cada parte), autocontenidos y ejecutables.
- Requisitos mínimos que debe incluir cada notebook:
	- **Sección de configuración** clara (imports, credenciales, modelo a usar)
	- **Código reproducible y documentado** con comentarios explicativos
	- **Salidas visibles** de todas las ejecuciones (no limpies los outputs del notebook)
	- **Análisis y reflexiones propias** escritas con tus palabras, no copiadas de la teoría
	- **Markdown bien estructurado** con títulos, subtítulos y secciones claras que faciliten la lectura